# 03. Training & Evaluation (Image Cell)

**Phase 2**: 폴루션 데이터셋 × 모델 5개 학습 → accuracy 측정.

체크포인트: (dataset, polluter, level, model) 단위로 skip.
GPU 시간 변수 — Colab Pro+ 권장.

---

In [ ]:
# ============================================================
# 0-1. 환경 + 의존성
# ============================================================
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from time import time

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image'
POLLUTED_DIR = f'{BASE}/data/image_polluted'

if BASE not in sys.path: sys.path.insert(0, BASE)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
%pip install -q timm

In [ ]:
# ============================================================
# 0-2. 사전등록 + 모델 정의
# ============================================================
DATASETS = {
    'CIFAR10': {'n_classes': 10, 'image_size': 32, 'channels': 3},
    'FashionMNIST': {'n_classes': 10, 'image_size': 28, 'channels': 1},
    'Flowers102': {'n_classes': 102, 'image_size': 224, 'channels': 3},
}
MODEL_NAMES = ['ResNet18', 'EfficientNetB0', 'MobileNetV3small', 'ViTTiny', 'CNNSimple']
EPOCHS = 30  # 정식 (사전등록 ADR-014)
BATCH_SIZE = 128
LR = 1e-3


def get_model(model_name, n_classes, in_channels=3):
    if model_name == 'ResNet18':
        m = tvm.resnet18(weights=None)
        if in_channels != 3:
            m.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
        return m
    if model_name == 'EfficientNetB0':
        m = tvm.efficientnet_b0(weights=None)
        if in_channels != 3:
            m.features[0][0] = nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1, bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
        return m
    if model_name == 'MobileNetV3small':
        m = tvm.mobilenet_v3_small(weights=None)
        if in_channels != 3:
            m.features[0][0] = nn.Conv2d(in_channels, 16, kernel_size=3, stride=2, padding=1, bias=False)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, n_classes)
        return m
    if model_name == 'ViTTiny':
        import timm
        return timm.create_model('vit_tiny_patch16_224', pretrained=False,
                                 num_classes=n_classes, in_chans=in_channels)
    if model_name == 'CNNSimple':
        return nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes),
        )

print('모델 정의 완료')

In [ ]:
# ============================================================
# 0-3. Dataset wrapper (numpy → tensor with transform)
# ============================================================
class NumpyDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        from PIL import Image
        img = self.images[i]
        if isinstance(img, np.ndarray):
            arr = img.squeeze() if img.ndim == 3 and img.shape[-1] == 1 else img
            img = Image.fromarray(arr) if arr.ndim == 2 else Image.fromarray(arr[..., :3])
        if img.mode != 'RGB':
            img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[i])


def get_transform(image_size=224):
    return T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def train_eval(model, train_ds, test_ds, epochs=EPOCHS):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
    # eval
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
    return correct / total

print('학습 함수 정의 완료')

## 1. 실험 목록 + 체크포인트

In [ ]:
# ============================================================
# 1-1. 실험 목록 스캔 (POLLUTED_DIR + clean baseline)
# ============================================================
def load_train_images(ds_name):
    """clean train (sample_cap 적용된 numpy arrays)."""
    raw_path = f'{DATA_DIR}/{ds_name}'
    if ds_name == 'CIFAR10':
        ds = torchvision.datasets.CIFAR10(raw_path, train=True, download=True)
    elif ds_name == 'FashionMNIST':
        ds = torchvision.datasets.FashionMNIST(raw_path, train=True, download=True)
    elif ds_name == 'Flowers102':
        ds = torchvision.datasets.Flowers102(raw_path, split='train', download=True)
    images, labels = [], []
    rng = np.random.RandomState(1)
    idx = rng.permutation(len(ds))[:5000]
    for i in idx:
        img, lbl = ds[i]
        images.append(np.array(img)); labels.append(int(lbl))
    return images, labels


def load_test_images(ds_name):
    raw_path = f'{DATA_DIR}/{ds_name}'
    if ds_name == 'CIFAR10':
        return torchvision.datasets.CIFAR10(raw_path, train=False, download=True)
    if ds_name == 'FashionMNIST':
        return torchvision.datasets.FashionMNIST(raw_path, train=False, download=True)
    if ds_name == 'Flowers102':
        return torchvision.datasets.Flowers102(raw_path, split='test', download=True)


def load_polluted(ds_name, polluter, level):
    npz = np.load(f'{POLLUTED_DIR}/{ds_name}/{polluter}_{int(level*100)}/data.npz', allow_pickle=True)
    return list(npz['images']), npz['labels'].tolist()


# 실험 목록
experiments = []
for ds_name in DATASETS:
    # baseline
    experiments.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0})
    # 폴루션
    pol_dir = f'{POLLUTED_DIR}/{ds_name}'
    if os.path.isdir(pol_dir):
        for folder in sorted(os.listdir(pol_dir)):
            if not os.path.isfile(f'{pol_dir}/{folder}/data.npz'):
                continue
            parts = folder.rsplit('_', 1)
            experiments.append({'dataset': ds_name, 'polluter': parts[0], 'level': int(parts[1])/100})

print(f'실험 목록: {len(experiments)}건 × 모델 {len(MODEL_NAMES)}개 = {len(experiments)*len(MODEL_NAMES)}회 학습')

In [ ]:
# ============================================================
# 2-1. 학습 루프 (체크포인트 지원)
# ============================================================
perf_path = f'{RESULTS_DIR}/model_performance_image.csv'

if os.path.isfile(perf_path):
    df_perf = pd.read_csv(perf_path)
    existing_keys = set(df_perf.apply(lambda r: f"{r['dataset']}|{r['polluter']}|{r['level']}|{r['model']}", axis=1))
    perf_rows = df_perf.to_dict('records')
    print(f'기존 결과 {len(perf_rows)}건 로드')
else:
    existing_keys, perf_rows = set(), []

total_start = time()
completed = skipped = 0
errors = []

for i, exp in enumerate(experiments):
    ds_name = exp['dataset']; meta = DATASETS[ds_name]
    # train images
    if exp['polluter'] == 'none':
        try: train_images, train_labels = load_train_images(ds_name)
        except Exception as e: print(f'load fail {ds_name}: {e}'); continue
    else:
        try: train_images, train_labels = load_polluted(ds_name, exp['polluter'], exp['level'])
        except Exception as e: print(f'load fail {ds_name}/{exp["polluter"]}_{int(exp["level"]*100)}: {e}'); continue

    # test (clean)
    test_raw = load_test_images(ds_name)
    test_images, test_labels = [], []
    for img, lbl in test_raw:
        test_images.append(np.array(img)); test_labels.append(int(lbl))

    transform = get_transform(image_size=224)
    train_ds = NumpyDataset(train_images, train_labels, transform)
    test_ds = NumpyDataset(test_images, test_labels, transform)

    for model_name in MODEL_NAMES:
        key = f"{ds_name}|{exp['polluter']}|{exp['level']}|{model_name}"
        if key in existing_keys:
            skipped += 1; continue
        try:
            t0 = time()
            model = get_model(model_name, meta['n_classes'])
            acc = train_eval(model, train_ds, test_ds, epochs=EPOCHS)
            elapsed = time() - t0
            row = {'dataset': ds_name, 'polluter': exp['polluter'], 'level': exp['level'],
                   'model': model_name, 'accuracy': round(acc, 4), 'epochs': EPOCHS}
            perf_rows.append(row); existing_keys.add(key); completed += 1
            print(f'  [{i+1}/{len(experiments)}] {ds_name}/{exp["polluter"]}_{int(exp["level"]*100)}/{model_name} acc={acc:.4f} ({elapsed:.0f}s)')
        except Exception as e:
            errors.append({'key': key, 'error': str(e)})
            print(f'  [{i+1}] {ds_name}/{model_name} ERROR: {e}')
        # 중간 저장
        if (completed + skipped) % 5 == 0:
            pd.DataFrame(perf_rows).to_csv(perf_path, index=False)

pd.DataFrame(perf_rows).to_csv(perf_path, index=False)
print(f'\n학습 완료: 완료={completed}, 스킵={skipped}, 에러={len(errors)} ({time()-total_start:.0f}s)')